# MINERA Goiás — Motor Econômico-Energético

## Objetivo

Este notebook implementa a versão demonstrativa e reproduzível do motor econômico-energético do projeto MINERA Goiás. Ele projeta a produção mineral e a demanda de energia elétrica de Goiás entre 2027 e 2040, por mineral e cenário.

Os dados desta versão são inteiramente sintéticos (`estimated_demo`). Eles existem apenas para testar a estrutura, as regras de cenário, os controles de consistência e os arquivos de saída. Quando a equipe disponibilizar dados reais, os arquivos de entrada deverão manter o mesmo contrato de dados.

## Execução no repositório

O notebook foi preparado para ser executado a partir da raiz do repositório. Os dados de demonstração devem estar em `Squad 2/data/demo/` e os resultados são gravados em `Squad 2/outputs/demo/`.

Não há carregamento manual de arquivos nesta versão. Isso permite que o mesmo notebook seja executado localmente, no Google Colab após clonar o repositório, e automaticamente pelo GitHub Actions.

## Lógica do modelo

1. Lê minerais, produção histórica, projetos, intensidades energéticas, parâmetros de cenário e regras de projetos.
2. Constrói o baseline de 2025 por mineral e base de produção.
3. Projeta a produção das operações existentes com uma taxa anual parametrizável.
4. Incorpora projetos conforme capacidade, estágio, utilização, atraso e fator de captura de mercado.
5. Calcula a demanda de energia como `produção projetada × intensidade energética`.
6. Agrega os resultados por mineral para o total anual de Goiás.
7. Executa análises de sensibilidade para atraso de projetos, utilização e eficiência energética.
8. Verifica cobertura, valores ausentes, não negatividade e reconciliação dos totais antes de exportar os resultados.

## Observação metodológica

O pipeline físico de operações e projetos é o núcleo do motor. Uma camada futura de demanda/elasticidade poderá ser adicionada como explicação macroeconômica ou limite de mercado, sem substituir a projeção física de oferta.


In [ ]:
from pathlib import Path


def encontrar_raiz_do_projeto():
    """Localiza a raiz do repositório a partir do diretório atual."""
    diretorio_atual = Path.cwd().resolve()
    for candidato in [diretorio_atual, *diretorio_atual.parents]:
        if (candidato / "Squad 2").exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível localizar a pasta 'Squad 2'. "
        "Execute o notebook a partir de uma cópia clonada do repositório."
    )


PROJECT_ROOT = encontrar_raiz_do_projeto()
INPUT_DIR = PROJECT_ROOT / "Squad 2" / "data" / "demo"
OUTPUT_DIR = PROJECT_ROOT / "Squad 2" / "outputs" / "demo"

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Pasta de entrada: {INPUT_DIR}")
print(f"Pasta de saída: {OUTPUT_DIR}")


In [ ]:
REQUIRED_INPUT_FILES = {
    "minerals_demo.csv",
    "production_history_demo.csv",
    "projects_demo.csv",
    "energy_intensity_demo.csv",
    "scenario_parameters_demo.csv",
    "scenario_project_rules_demo.csv",
}

missing_files = sorted(
    file_name for file_name in REQUIRED_INPUT_FILES if not (INPUT_DIR / file_name).is_file()
)

if missing_files:
    raise FileNotFoundError(
        "Arquivos de entrada ausentes em "
        f"{INPUT_DIR}: {', '.join(missing_files)}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Arquivos de entrada encontrados:")
for file_name in sorted(REQUIRED_INPUT_FILES):
    print(f"- {file_name}")


In [ ]:
import pandas as pd

minerals = pd.read_csv(INPUT_DIR / "minerals_demo.csv", encoding="utf-8-sig")
production = pd.read_csv(INPUT_DIR / "production_history_demo.csv", encoding="utf-8-sig")
projects = pd.read_csv(INPUT_DIR / "projects_demo.csv", encoding="utf-8-sig")
intensity = pd.read_csv(INPUT_DIR / "energy_intensity_demo.csv", encoding="utf-8-sig")
parameters = pd.read_csv(INPUT_DIR / "scenario_parameters_demo.csv", encoding="utf-8-sig")
project_rules = pd.read_csv(INPUT_DIR / "scenario_project_rules_demo.csv", encoding="utf-8-sig")

print("Minerais:", len(minerals))
print("Linhas de produção histórica:", len(production))
print("Projetos:", len(projects))
print("Linhas de intensidade energética:", len(intensity))
print("Parâmetros de cenário:", len(parameters))

display(production.head())
display(projects.head())


In [ ]:
production_2025 = production[production["year"] == 2025].copy()
intensity_2025 = intensity[intensity["year"] == 2025].copy()

baseline_detail = production_2025.merge(
    intensity_2025[
        ["mineral_id", "operation_id", "production_basis", "energy_intensity_mwh_t"]
    ],
    on=["mineral_id", "operation_id", "production_basis"],
    how="inner"
)

baseline_detail["energy_mwh"] = (
    baseline_detail["production_t"] * baseline_detail["energy_intensity_mwh_t"]
)

baseline = (
    baseline_detail
    .groupby(["mineral_id", "production_basis"], as_index=False)
    .agg(
        baseline_production_t=("production_t", "sum"),
        baseline_energy_mwh=("energy_mwh", "sum")
    )
)

baseline["baseline_intensity_mwh_t"] = (
    baseline["baseline_energy_mwh"] / baseline["baseline_production_t"]
)

baseline = baseline.merge(
    minerals[["mineral_id", "mineral_name"]],
    on="mineral_id",
    how="left"
)

display(
    baseline[
        [
            "mineral_id",
            "mineral_name",
            "production_basis",
            "baseline_production_t",
            "baseline_intensity_mwh_t"
        ]
    ]
)


In [ ]:
growth_parameters = parameters[
    parameters["parameter_name"] == "existing_production_growth_rate"
].copy()

existing_projection = baseline.merge(
    growth_parameters[
        ["mineral_id", "year", "scenario", "parameter_value"]
    ],
    on="mineral_id",
    how="inner"
)

existing_projection = existing_projection.sort_values(
    ["mineral_id", "scenario", "year"]
).copy()

# Em 2027, aplica-se a taxa duas vezes: 2025→2026 e 2026→2027.
existing_projection["annual_growth_factor"] = (
    1 + existing_projection["parameter_value"]
)

existing_projection["growth_factor"] = existing_projection[
    "annual_growth_factor"
]

existing_projection.loc[
    existing_projection["year"] == 2027,
    "growth_factor"
] = existing_projection.loc[
    existing_projection["year"] == 2027,
    "annual_growth_factor"
] ** 2

existing_projection["cumulative_growth_factor"] = (
    existing_projection
    .groupby(["mineral_id", "scenario"])["growth_factor"]
    .cumprod()
)

existing_projection["existing_production_t"] = (
    existing_projection["baseline_production_t"]
    * existing_projection["cumulative_growth_factor"]
)

display(
    existing_projection[
        existing_projection["year"].isin([2027, 2040])
    ][
        [
            "mineral_id",
            "mineral_name",
            "year",
            "scenario",
            "existing_production_t"
        ]
    ]
)

In [ ]:
# Seleciona os parâmetros anuais de utilização e captura de mercado dos projetos.
utilization_parameters = parameters[
    parameters["parameter_name"] == "project_utilization_rate"
][
    ["mineral_id", "year", "scenario", "parameter_value"]
].rename(columns={"parameter_value": "project_utilization_rate"})

market_capture_parameters = parameters[
    parameters["parameter_name"] == "market_capture_factor"
][
    ["mineral_id", "year", "scenario", "parameter_value"]
].rename(columns={"parameter_value": "market_capture_factor"})

# Converte explicitamente a regra de inclusão em valor booleano.
project_rules["include_project"] = (
    project_rules["include_project"].astype(str).str.lower().eq("true")
)

# Cria uma linha por projeto, ano e cenário.
project_calculation = (
    existing_projection[["mineral_id", "year", "scenario"]]
    .merge(
        projects[
            [
                "project_id",
                "mineral_id",
                "capacity_tpy",
                "start_year",
                "project_stage",
                "production_basis",
            ]
        ],
        on="mineral_id",
        how="left",
    )
    .merge(
        project_rules[
            ["scenario", "project_stage", "include_project", "start_delay_years"]
        ],
        on=["scenario", "project_stage"],
        how="left",
    )
    .merge(
        utilization_parameters,
        on=["mineral_id", "year", "scenario"],
        how="left",
    )
    .merge(
        market_capture_parameters,
        on=["mineral_id", "year", "scenario"],
        how="left",
    )
)

required_project_fields = [
    "include_project",
    "start_delay_years",
    "project_utilization_rate",
    "market_capture_factor",
]
if project_calculation[required_project_fields].isna().any().any():
    raise ValueError("Há regras ou parâmetros ausentes para o cálculo dos projetos.")

# Um projeto produz após o ano de início, acrescido do atraso do cenário.
project_calculation["effective_start_year"] = (
    project_calculation["start_year"] + project_calculation["start_delay_years"]
)

project_calculation["project_production_t"] = 0.0
active_project = (
    project_calculation["include_project"]
    & (project_calculation["year"] >= project_calculation["effective_start_year"])
)

# Produção do projeto = capacidade × utilização × captura de mercado.
project_calculation.loc[active_project, "project_production_t"] = (
    project_calculation.loc[active_project, "capacity_tpy"]
    * project_calculation.loc[active_project, "project_utilization_rate"]
    * project_calculation.loc[active_project, "market_capture_factor"]
)

project_total = (
    project_calculation.groupby(["mineral_id", "year", "scenario"], as_index=False)
    .agg(project_production_t=("project_production_t", "sum"))
)

# Produção total = operações existentes + projetos que entraram em operação.
production_projection = existing_projection.merge(
    project_total,
    on=["mineral_id", "year", "scenario"],
    how="left",
)
production_projection["project_production_t"] = (
    production_projection["project_production_t"].fillna(0)
)
production_projection["projected_production_t"] = (
    production_projection["existing_production_t"]
    + production_projection["project_production_t"]
)

display(
    production_projection[
        production_projection["year"].isin([2027, 2030, 2040])
    ][
        [
            "mineral_id",
            "mineral_name",
            "year",
            "scenario",
            "existing_production_t",
            "project_production_t",
            "projected_production_t",
        ]
    ]
)


In [ ]:
efficiency_parameters = parameters[
    parameters["parameter_name"] == "annual_efficiency_improvement_rate"
][
    ["mineral_id", "year", "scenario", "parameter_value"]
].rename(columns={"parameter_value": "annual_efficiency_improvement_rate"})

energy_projection = production_projection.merge(
    efficiency_parameters,
    on=["mineral_id", "year", "scenario"],
    how="left"
)

energy_projection = energy_projection.sort_values(
    ["mineral_id", "scenario", "year"]
).copy()

# A intensidade diminui conforme o ganho de eficiência energética.
energy_projection["annual_efficiency_factor"] = (
    1 - energy_projection["annual_efficiency_improvement_rate"]
)

energy_projection["efficiency_factor"] = (
    energy_projection["annual_efficiency_factor"]
)

# Entre 2025 e 2027, consideram-se dois anos de melhoria.
energy_projection.loc[
    energy_projection["year"] == 2027,
    "efficiency_factor"
] = energy_projection.loc[
    energy_projection["year"] == 2027,
    "annual_efficiency_factor"
] ** 2

energy_projection["cumulative_efficiency_factor"] = (
    energy_projection
    .groupby(["mineral_id", "scenario"])["efficiency_factor"]
    .cumprod()
)

energy_projection["energy_intensity_mwh_t"] = (
    energy_projection["baseline_intensity_mwh_t"]
    * energy_projection["cumulative_efficiency_factor"]
)

energy_projection["energy_demand_mwh"] = (
    energy_projection["projected_production_t"]
    * energy_projection["energy_intensity_mwh_t"]
)

final_output = energy_projection[
    [
        "mineral_id",
        "mineral_name",
        "production_basis",
        "year",
        "scenario",
        "projected_production_t",
        "energy_intensity_mwh_t",
        "energy_demand_mwh"
    ]
].copy()

display(
    final_output[
        (final_output["mineral_id"] == "MIN_001")
        & (final_output["year"].isin([2027, 2030, 2040]))
    ]
)

In [ ]:
goias_total = (
    final_output
    .groupby(["year", "scenario"], as_index=False)
    .agg(
        total_projected_production_t=("projected_production_t", "sum"),
        total_energy_demand_mwh=("energy_demand_mwh", "sum")
    )
)

goias_total["state_energy_intensity_mwh_t"] = (
    goias_total["total_energy_demand_mwh"]
    / goias_total["total_projected_production_t"]
)

display(goias_total)

In [ ]:
expected_years = set(range(2027, 2041))
expected_scenarios = {"conservador", "referencia", "expansao"}

# Cada combinação mineral-cenário deve conter todos os anos projetados.
coverage = (
    final_output
    .groupby(["mineral_id", "scenario"])["year"]
    .agg(["count", "min", "max"])
    .reset_index()
)

all_years_present = (
    (coverage["count"] == 14).all()
    and (coverage["min"] == 2027).all()
    and (coverage["max"] == 2040).all()
)

# O total de Goiás deve coincidir exatamente com a soma dos minerais.
recalculated_total = (
    final_output
    .groupby(["year", "scenario"], as_index=False)
    .agg(
        production_from_minerals=("projected_production_t", "sum"),
        energy_from_minerals=("energy_demand_mwh", "sum")
    )
)

reconciliation = goias_total.merge(
    recalculated_total,
    on=["year", "scenario"],
    how="left"
)

reconciliation["production_difference"] = (
    reconciliation["total_projected_production_t"]
    - reconciliation["production_from_minerals"]
)

reconciliation["energy_difference"] = (
    reconciliation["total_energy_demand_mwh"]
    - reconciliation["energy_from_minerals"]
)

check_results = pd.DataFrame({
    "controle": [
        "Número de linhas detalhadas",
        "Número de linhas totais Goiás",
        "Anos completos para cada mineral e cenário",
        "Cenários corretos",
        "Valores ausentes",
        "Produção e energia não negativas",
        "Total Goiás = soma dos minerais"
    ],
    "resultado": [
        len(final_output) == len(existing_projection),
        len(goias_total) == len(expected_years) * len(expected_scenarios),
        all_years_present,
        set(final_output["scenario"]) == expected_scenarios,
        final_output.isna().sum().sum() == 0,
        (
            (final_output["projected_production_t"] >= 0).all()
            and (final_output["energy_intensity_mwh_t"] > 0).all()
            and (final_output["energy_demand_mwh"] >= 0).all()
        ),
        (
            reconciliation["production_difference"].abs().max() < 0.0001
            and reconciliation["energy_difference"].abs().max() < 0.0001
        )
    ]
})

display(check_results)
display(reconciliation)

if not check_results["resultado"].all():
    failed_checks = check_results.loc[~check_results["resultado"], "controle"].tolist()
    raise AssertionError(f"Controles de consistência reprovados: {failed_checks}")

In [ ]:
# ANÁLISE DE SENSIBILIDADE PARAMETRIZÁVEL
# Esta célula mantém o cenário de referência e altera uma hipótese por vez.
# Os intervalos podem ser modificados sem alterar a lógica do modelo.

scenario_base = "referencia"

# Atrasos adicionais: de 0 a 24 meses, em intervalos de 6 meses.
delay_months_values = list(range(0, 25, 6))

# Variação geral da utilização: -20 a +10 p.p., em passos de 1 p.p.
# A utilização final é sempre limitada entre 0% e 100%.
utilization_adjustment_pp_values = list(range(-20, 11, 1))

# Variação do ganho anual de eficiência: -2,0 a +2,0 p.p.,
# em passos de 0,1 p.p.
# Valor positivo = maior redução anual da intensidade energética.
efficiency_adjustment_pp_values = [
    round(value / 10, 1) for value in range(-20, 21)
]

# Resultado original do cenário de referência.
reference_output = final_output[
    final_output["scenario"] == scenario_base
].copy()

# Produção das operações existentes no cenário de referência.
reference_existing = production_projection[
    production_projection["scenario"] == scenario_base
][
    ["mineral_id", "year", "existing_production_t"]
].copy()

# Detalhe dos projetos no cenário de referência.
reference_projects = project_calculation[
    project_calculation["scenario"] == scenario_base
].copy()

# Intensidade energética original para testes que não alteram eficiência.
reference_intensity = reference_output[
    [
        "mineral_id",
        "year",
        "mineral_name",
        "production_basis",
        "energy_intensity_mwh_t"
    ]
].copy()


def calculate_project_sensitivity(
    test_group,
    parameter_name,
    parameter_value,
    additional_delay_months=0,
    utilization_adjustment_pp=0
):
    """
    Recalcula a produção dos projetos.
    A produção existente e a intensidade energética permanecem iguais
    ao cenário de referência.
    """

    test = reference_projects.copy()

    # Converte o atraso adicional em anos completos e meses restantes.
    full_delay_years = additional_delay_months // 12
    remaining_delay_months = additional_delay_months % 12

    # Ano de início após o atraso adicional.
    test["effective_start_year_test"] = (
        test["effective_start_year"] + full_delay_years
    )

    # Aplica variação geral à utilização e limita o resultado entre 0% e 100%.
    test["project_utilization_rate_test"] = (
        test["project_utilization_rate"]
        + utilization_adjustment_pp / 100
    ).clip(lower=0, upper=1)

    # Um projeto produz somente após o ano efetivo de entrada.
    project_active = (
        test["include_project"]
        & (test["year"] >= test["effective_start_year_test"])
    )

    test["operation_share"] = 0.0
    test.loc[project_active, "operation_share"] = 1.0

    # Atrasos de 6 ou 18 meses geram produção parcial no primeiro ano ativo.
    if remaining_delay_months > 0:
        partial_first_year = (
            project_active
            & (test["year"] == test["effective_start_year_test"])
        )

        test.loc[partial_first_year, "operation_share"] = (
            1 - remaining_delay_months / 12
        )

    # Produção = capacidade × utilização × captura de mercado
    # × parcela do ano em operação.
    test["project_production_t_test"] = (
        test["capacity_tpy"]
        * test["project_utilization_rate_test"]
        * test["market_capture_factor"]
        * test["operation_share"]
    )

    project_total = (
        test
        .groupby(["mineral_id", "year"], as_index=False)
        .agg(project_production_t=("project_production_t_test", "sum"))
    )

    result = reference_existing.merge(
        project_total,
        on=["mineral_id", "year"],
        how="left"
    ).merge(
        reference_intensity,
        on=["mineral_id", "year"],
        how="left"
    )

    result["project_production_t"] = (
        result["project_production_t"].fillna(0)
    )

    result["projected_production_t"] = (
        result["existing_production_t"]
        + result["project_production_t"]
    )

    result["energy_demand_mwh"] = (
        result["projected_production_t"]
        * result["energy_intensity_mwh_t"]
    )

    result["test_group"] = test_group
    result["parameter_name"] = parameter_name
    result["parameter_value"] = parameter_value

    return result


# 1. Sensibilidade ao atraso dos projetos.
delay_results = []

for delay_months in delay_months_values:
    delay_results.append(
        calculate_project_sensitivity(
            test_group="atraso_de_projetos",
            parameter_name="atraso_adicional_meses",
            parameter_value=delay_months,
            additional_delay_months=delay_months
        )
    )

delay_results = pd.concat(delay_results, ignore_index=True)


# 2. Sensibilidade à utilização geral dos projetos.
utilization_results = []

for utilization_adjustment_pp in utilization_adjustment_pp_values:
    utilization_results.append(
        calculate_project_sensitivity(
            test_group="utilizacao_dos_projetos",
            parameter_name="variacao_utilizacao_pp",
            parameter_value=utilization_adjustment_pp,
            utilization_adjustment_pp=utilization_adjustment_pp
        )
    )

utilization_results = pd.concat(utilization_results, ignore_index=True)


# 3. Sensibilidade à eficiência energética.
# A produção permanece igual; mudam a intensidade e a demanda de energia.
efficiency_parameters = parameters[
    (parameters["scenario"] == scenario_base)
    & (parameters["parameter_name"] == "annual_efficiency_improvement_rate")
][
    ["mineral_id", "year", "parameter_value"]
].rename(columns={"parameter_value": "efficiency_rate"})

efficiency_results = []

for efficiency_adjustment_pp in efficiency_adjustment_pp_values:

    test = reference_output[
        [
            "mineral_id",
            "mineral_name",
            "production_basis",
            "year",
            "projected_production_t"
        ]
    ].copy()

    test = test.merge(
        baseline[
            [
                "mineral_id",
                "production_basis",
                "baseline_intensity_mwh_t"
            ]
        ],
        on=["mineral_id", "production_basis"],
        how="left"
    ).merge(
        efficiency_parameters,
        on=["mineral_id", "year"],
        how="left"
    ).sort_values(["mineral_id", "year"])

    # A taxa original representa redução da intensidade energética.
    # Valor positivo no teste aumenta essa redução; valor negativo a diminui.
    test["annual_efficiency_factor"] = (
        1
        - test["efficiency_rate"]
        - efficiency_adjustment_pp / 100
    )

    test["efficiency_factor"] = test["annual_efficiency_factor"]

    # Em 2027 o fator é aplicado duas vezes: 2025→2026 e 2026→2027.
    test.loc[
        test["year"] == 2027,
        "efficiency_factor"
    ] = test.loc[
        test["year"] == 2027,
        "annual_efficiency_factor"
    ] ** 2

    test["cumulative_efficiency_factor"] = (
        test
        .groupby("mineral_id")["efficiency_factor"]
        .cumprod()
    )

    test["energy_intensity_mwh_t"] = (
        test["baseline_intensity_mwh_t"]
        * test["cumulative_efficiency_factor"]
    )

    test["energy_demand_mwh"] = (
        test["projected_production_t"]
        * test["energy_intensity_mwh_t"]
    )

    test["test_group"] = "eficiencia_energetica"
    test["parameter_name"] = "variacao_melhoria_eficiencia_pp"
    test["parameter_value"] = efficiency_adjustment_pp

    efficiency_results.append(test)

efficiency_results = pd.concat(efficiency_results, ignore_index=True)


# Junta todas as análises: mineral × ano × valor do parâmetro.
sensitivity_results = pd.concat(
    [
        delay_results,
        utilization_results,
        efficiency_results
    ],
    ignore_index=True
)

# Agrega cada teste para o total anual de Goiás.
sensitivity_goias_total = (
    sensitivity_results
    .groupby(
        ["test_group", "parameter_name", "parameter_value", "year"],
        as_index=False
    )
    .agg(
        projected_production_t=("projected_production_t", "sum"),
        energy_demand_mwh=("energy_demand_mwh", "sum")
    )
)

# Adiciona o cenário de referência para calcular diferenças.
reference_goias_total = goias_total[
    goias_total["scenario"] == scenario_base
][
    ["year", "total_projected_production_t", "total_energy_demand_mwh"]
].copy()

sensitivity_goias_total = sensitivity_goias_total.merge(
    reference_goias_total,
    on="year",
    how="left"
)

sensitivity_goias_total["production_difference_t"] = (
    sensitivity_goias_total["projected_production_t"]
    - sensitivity_goias_total["total_projected_production_t"]
)

sensitivity_goias_total["energy_difference_mwh"] = (
    sensitivity_goias_total["energy_demand_mwh"]
    - sensitivity_goias_total["total_energy_demand_mwh"]
)

sensitivity_goias_total["production_difference_pct"] = (
    100
    * sensitivity_goias_total["production_difference_t"]
    / sensitivity_goias_total["total_projected_production_t"]
)

sensitivity_goias_total["energy_difference_pct"] = (
    100
    * sensitivity_goias_total["energy_difference_mwh"]
    / sensitivity_goias_total["total_energy_demand_mwh"]
)

# Mostra a tabela final para os anos-chave.
sensitivity_summary = sensitivity_goias_total[
    sensitivity_goias_total["year"].isin([2030, 2035, 2040])
].sort_values(
    ["test_group", "parameter_value", "year"]
).copy()

display(sensitivity_summary.round(2))

In [ ]:
import shutil

# Cria a pasta que receberá todos os resultados do modelo.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Resultado principal: produção e demanda de energia por mineral, ano e cenário.
output_by_mineral = final_output.copy()
output_by_mineral["data_nature"] = "estimated_demo"
output_by_mineral["source_id"] = "SRC_DEMO_001"

# 2. Resultado agregado: total anual de Goiás por cenário.
output_goias_total = goias_total.copy()
output_goias_total["data_nature"] = "estimated_demo"
output_goias_total["source_id"] = "SRC_DEMO_001"

# 3. Rastreabilidade: contribuição de cada projeto.
output_by_project = project_calculation[
    [
        "project_id",
        "mineral_id",
        "year",
        "scenario",
        "project_stage",
        "start_year",
        "effective_start_year",
        "capacity_tpy",
        "production_basis",
        "project_utilization_rate",
        "market_capture_factor",
        "project_production_t",
    ]
].copy()
output_by_project["data_nature"] = "estimated_demo"
output_by_project["source_id"] = "SRC_DEMO_001"

# 4. Sensibilidade detalhada: mineral, ano e valor de cada parâmetro testado.
output_sensitivity_detail = sensitivity_results.copy()
output_sensitivity_detail["data_nature"] = "estimated_demo"
output_sensitivity_detail["source_id"] = "SRC_DEMO_001"

# 5. Sensibilidade agregada: total anual de Goiás para cada valor testado.
output_sensitivity_goias = sensitivity_goias_total.copy()
output_sensitivity_goias["data_nature"] = "estimated_demo"
output_sensitivity_goias["source_id"] = "SRC_DEMO_001"

# 6. Resumo da sensibilidade para os anos-chave 2030, 2035 e 2040.
output_sensitivity_summary = sensitivity_summary.copy()
output_sensitivity_summary["data_nature"] = "estimated_demo"
output_sensitivity_summary["source_id"] = "SRC_DEMO_001"

# Salva em UTF-8 com BOM para preservar acentos em leitores de planilha.
output_by_mineral.to_csv(
    OUTPUT_DIR / "projecao_por_mineral_demo.csv", index=False, encoding="utf-8-sig"
)
output_goias_total.to_csv(
    OUTPUT_DIR / "projecao_total_goias_demo.csv", index=False, encoding="utf-8-sig"
)
output_by_project.to_csv(
    OUTPUT_DIR / "contribuicao_projetos_demo.csv", index=False, encoding="utf-8-sig"
)
output_sensitivity_detail.to_csv(
    OUTPUT_DIR / "sensibilidade_detalhada_demo.csv", index=False, encoding="utf-8-sig"
)
output_sensitivity_goias.to_csv(
    OUTPUT_DIR / "sensibilidade_total_goias_demo.csv", index=False, encoding="utf-8-sig"
)
output_sensitivity_summary.to_csv(
    OUTPUT_DIR / "sensibilidade_resumo_demo.csv", index=False, encoding="utf-8-sig"
)

# Cria um arquivo ZIP reproduzível com todos os resultados do demo.
# O arquivo temporário fica fora da pasta de saída para não ser incluído nele mesmo.
zip_file = OUTPUT_DIR / "MINERA_Goias_outputs_demo.zip"
temporary_zip_base = PROJECT_ROOT / "MINERA_Goias_outputs_demo_temp"
temporary_zip_file = temporary_zip_base.with_suffix(".zip")
if zip_file.exists():
    zip_file.unlink()
if temporary_zip_file.exists():
    temporary_zip_file.unlink()
shutil.make_archive(str(temporary_zip_base), "zip", root_dir=OUTPUT_DIR)
temporary_zip_file.replace(zip_file)

print("Resultados criados:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name}")
